# 03 Gold

Builds the tables the front end reads. Covers 1990 to 2020.

1. **gold_totals_by_year:** image counts per year, gender, line and shot type. The denominators, so busy years don't fake a trend.
2. **gold_tag_by_year:** for each clean tag and year, the count and share, split the same way.
3. **gold_house_by_season:** each collection, its look count, and one hero image.
4. Validation checks on all three.

### Totals by year
Counts every image per year, gender, line and shot type. The front end divides tag counts by these, using the same toggles, so a busy year like 2016 doesn't make every trend look bigger.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

e = (spark.table("workspace.silhouette_archive.silver_runway_enriched")
     .filter(F.col("year").between(1990, 2020)))

totals = e.groupBy("year", "gender", "line", "shot_type").agg(F.count("*").alias("total_images"))

totals.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.gold_totals_by_year")
display(spark.table("workspace.silhouette_archive.gold_totals_by_year").orderBy("year", "gender", "line", "shot_type"))

### Tags by year
For each clean tag, counts how many images have it per year, gender, line and shot type, then divides by the matching total from gold_totals_by_year to get a share.

**Note:** when combining toggles, always sum totals from gold_totals_by_year, not from this table. This table only has buckets where the tag appears, so summing its totals undercounts and inflates the share.

In [0]:
st = spark.table("workspace.silhouette_archive.silver_tags")
totals = spark.table("workspace.silhouette_archive.gold_totals_by_year")
tkeys = ["year", "gender", "line", "shot_type"]

tag_counts = (st.join(e.select("row_id", *tkeys), "row_id")
    .groupBy("tag", *tkeys)
    .agg(F.count("*").alias("tag_images")))

tag_by_year = (tag_counts.join(totals, tkeys)
    .withColumn("share", F.col("tag_images") / F.col("total_images")))

tag_by_year.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.gold_tag_by_year")
display(spark.table("workspace.silhouette_archive.gold_tag_by_year")
        .filter("tag = 'Coat'").orderBy(*tkeys))

### House by season
One row per collection: designer, year, season, gender and line, its look count, and one hero image.

**Hero rule:** the middle look of the show, preferring normal resolution. The first and last images are often the designer, thumbnails or the final bow, so the middle is almost always a real runway look.

**Rejected rules:**
- Highest aesthetic score: picked the designer's bow.
- First look in data order: often the designer or a thumbnail.

Low-res images rank last instead of being removed, so 2000 to 2005 collections still get a hero.

In [0]:
hkeys = ["designer", "year", "season", "gender", "line"]

ranked = (e.filter("shot_type = 'full_look'")
    .withColumn("pos", F.row_number().over(Window.partitionBy(*hkeys).orderBy("row_id")))
    .withColumn("n", F.count("*").over(Window.partitionBy(*hkeys)))
    .withColumn("dist", F.abs(F.col("pos") - (F.col("n") + 1) / 2)))

w = Window.partitionBy(*hkeys).orderBy(F.col("is_low_res").asc(), "dist", "row_id")

house = (ranked.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select(*hkeys, F.col("n").alias("looks"),
            F.col("row_id").alias("hero_row_id"), F.col("url").alias("hero_url")))

house.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.gold_house_by_season")
display(spark.table("workspace.silhouette_archive.gold_house_by_season")
        .filter(F.lower("designer").contains("yohji")).orderBy("year", "season"))

### Validate gold
- Totals add up to the number of silver images from 1990 to 2020
- Every share is above 0 and at most 1
- No missing years in the totals
- Every collection appears once and has a hero image

In [0]:
t = spark.table("workspace.silhouette_archive.gold_totals_by_year")
tg = spark.table("workspace.silhouette_archive.gold_tag_by_year")
h = spark.table("workspace.silhouette_archive.gold_house_by_season")

assert t.agg(F.sum("total_images")).first()[0] == e.count(), "Totals don't match silver"

bad = tg.filter((F.col("share") <= 0) | (F.col("share") > 1)).count()
assert bad == 0, f"{bad} shares outside 0 to 1"

missing = set(range(1990, 2021)) - {r.year for r in t.select("year").distinct().collect()}
assert not missing, f"Missing years: {missing}"

assert h.count() == h.select(*hkeys).distinct().count(), "Duplicate collections"
assert h.filter(F.col("hero_url").isNull()).count() == 0, "Collections without a hero image"

print(f"All checks passed: {t.count():,} total rows, {tg.count():,} tag rows, {h.count():,} collections")